# vLLM Support

## Summary

[vLLM](https://github.com/vllm-project/vllm) is a high-performance inference
library that uses PagedAttention, dynamic batching, and a custom execution
engine to serve HuggingFace-compatible language models at much higher
throughput than a plain `transformers` run. NNsight ships with a full vLLM
backend so you can observe and intervene on model internals without giving up
vLLM's speed.

This guide walks through all the ways to use the integration:

- Instantiating the `VLLM` wrapper (sync and async)
- Reading and writing module activations inside a trace
- Batching multiple prompts with `tracer.invoke()` loops
- The `logits` and `samples` eproperties for inspecting/modifying sampling
- Multi-token generation with `tracer.all()` and `tracer.iter[...]`
- Efficient activation extraction with `tracer.cache()`
- Async mode (`mode="async"`) for streaming per-request saves
- Tensor parallelism and the Ray distributed executor for multi-GPU / multi-node
- Known limitations and gotchas

## When to use

`VLLM` is the right tool when you need high-throughput generation or want to
run experiments over many prompts with the same interventions.

A few considerations before choosing `VLLM` over `LanguageModel`:

- **Text generation only.** NNsight supports vLLM's text-generation models
  (list [here](https://docs.vllm.ai/en/latest/models/supported_models/#text-generation)).
  Multimodal and image-generation models aren't supported yet.
- **vLLM results may differ from transformers.** vLLM's custom kernels,
  quantization defaults, and batching produce slightly different numerical
  outputs than `transformers`, so comparisons across the two backends are
  never exact.
- **No gradients.** vLLM's paged-attention path doesn't support autograd, so
  `VLLM` models can't be used for backward-pass experiments. Use
  `LanguageModel` if you need gradients.
- **One prompt per invoke.** Unlike `LanguageModel`, each `tracer.invoke(...)`
  in a vLLM trace must receive exactly one prompt. Batching is done by
  putting many invokes inside a single trace — vLLM's scheduler takes care of
  batching them on the GPU.

## Setup

Install NNsight alongside `vllm` and `triton`. `zstandard` is used by the
async backend for transport compression.

<pre><code>pip install "nnsight[vllm]" "vllm-lens>=1.1.0" zstandard
</code></pre>

In [1]:
import os
# Reduce vLLM logging
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"  
os.environ["VLLM_LOGGING_LEVEL"]   = "ERROR"
os.environ["GLOO_LOG_LEVEL"]       = "ERROR"

os.environ["HF_HOME"] = "/users/ychan19/data_sbach/ychan19/hf_cache"

# NNsight's vLLM backend patches the engine in-process, so run the sync engine
# core in-process (not in a subprocess), otherwise the interventions never fire.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

# The async engine near the end launches a worker subprocess. 
# Since the in-process sync engine initializes CUDA in this process,
# that subprocess must be spawned, not forked. 
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
import multiprocessing as mp
mp.set_start_method("spawn", force=True)

In [2]:
from IPython.display import clear_output
from pprint import pprint

## Instantiating a vLLM model

In [3]:
from nnsight.modeling.vllm import VLLM

model = VLLM(
    "Qwen/Qwen2.5-0.5B",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.2,
    dispatch=True,
    download_dir="/users/ychan19/data_sbach/ychan19/hf_cache",
    enable_prefix_caching=False
)

print(model)

/oscar/data/sbach/ychan19/envs/nnsight/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0

The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  6.59it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  6.51it/s]



Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): VocabParallelEmbedding(num_embeddings=151936, embedding_dim=896, org_vocab_size=151936, num_embeddings_padded=151936, tp_size=1)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (qkv_proj): QKVParallelLinear(in_features=896, output_features=1152, bias=True, tp_size=1, gather_output=False)
          (o_proj): RowParallelLinear(in_features=896, output_features=896, bias=False, tp_size=1, reduce_results=True)
          (rotary_emb): RotaryEmbedding(
            head_size=64, rotary_dim=64, max_position_embeddings=32768, base=1000000.0, is_neox_style=True
            (apply_rotary_emb): ApplyRotaryEmb(is_neox_style=True, enable_fp32_compute=False)
          )
          (attn): Attention(head_size=64, num_heads=14, num_kv_heads=2, scale=0.125, backend=FlashAttentionImpl)
        )
        (mlp): Qwen2MLP(
          (gate_up_proj): MergedColumnParallelLinear(in_features=

The wrapper looks and behaves like any other NNsight model: the underlying
vLLM engine is reachable via `model.vllm_entrypoint`, and the module envoys
(`model.model.layers[...]`, `model.lm_head`, etc.) are what you intervene on.

Two wrapper-only attributes are worth knowing about upfront:

- `model.logits` — the final logits tensor before sampling
- `model.samples` — the sampled token ids after temperature / top-p

We'll use both later on.

## Basic interventions

You can read or write any module's inputs/outputs inside a `model.trace(...)`
block, just like with `LanguageModel`:

In [4]:
prompt = "The Eiffel Tower is in the city of "

with model.trace(prompt, temperature=0.0, top_p=1) as tracer:
    # Read: middle-layer MLP output (a plain tensor)
    mlp_out = model.model.layers[6].mlp.output.save()

    # Write: zero the last token's hidden state going into lm_head. vLLM's fused
    # RMSNorm returns a (normalized, residual) tuple, so clone the normalized
    # stream, zero its last row, and assign the whole tuple back.
    norm_out = model.model.norm.output
    normed = model.model.norm.output[0].clone()
    normed[-1, :] = 0
    model.model.norm.output = (normed, norm_out[1])

    # Grab the first predicted token (changed by the write above)
    next_token = model.logits.argmax(dim=-1).save()

print("mlp_out shape:", mlp_out.shape)
print("predicted token:", repr(model.tokenizer.decode(next_token)))

Rendering prompts:   0%|                                                                           | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|█████████████| 1/1 [00:00<00:00,  3.07it/s, est. speed input: 34.09 toks/s, output: 49.58 toks/s]

mlp_out shape: torch.Size([11, 896])
predicted token: '!'


A few things to notice:

1. **Flat token layout.** vLLM concatenates all tokens from all prompts into a
   single `[total_tokens, hidden]` tensor — there's no batch dimension. That's
   why the slice above is `[-1, :]` and not `[:, -1, :]` like you'd write for
   `LanguageModel`. NNsight narrows the right slice for you automatically when
   you're inside a single-prompt invoke, so you still work in "per-request"
   coordinates.
2. **Clone, then swap the whole output.** vLLM executes inside
   `torch.inference_mode()`, so module outputs are read-only. Grab a clone,
   mutate it, and assign the whole value back through `.output` — NNsight picks
   up the assigned value and feeds it to the next layer. vLLM's fused RMSNorm
   returns a `(normalized, residual)` tuple, so we swap the tuple back as a
   whole. Prefer this attribute swap over an in-place `output[...] = ...` write:
   on the current vLLM build the in-place form applies to the forward pass but
   silently fails to return the `.save()`d values from that trace (you can get
   stale values with no error).
3. **Save works the same.** `.save()` (or `nns.save(...)`) marks a value to be
   transported back to your process once generation finishes. Anything not
   saved is discarded.

## Batching multiple prompts with invoke loops

With `LanguageModel` you can pass a list of prompts to a single invoke. With
vLLM you **can't** — each invoke is one request. Instead you put a loop of
`tracer.invoke(...)` calls inside a single `trace()` context:

In [5]:
prompts = [
    "The Eiffel Tower is in the city of",
    "Madison Square Garden is in the city of",
    "The Colosseum is in the city of",
]

with model.trace(temperature=0.0, top_p=1) as tracer:
    predictions = list().save()

    for prompt in prompts:
        with tracer.invoke(prompt):
            token_id = model.logits.argmax(dim=-1)
            predictions.append(model.tokenizer.decode(token_id))

for prompt, pred in zip(prompts, predictions):
    print(f"{prompt!r:<46} → {pred!r}")

Processed prompts:   0%|                       | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|███████████| 3/3 [00:00<00:00, 10.79it/s, est. speed input: 104.80 toks/s, output: 173.46 toks/s]

'The Eiffel Tower is in the city of'           → ' Paris'
'Madison Square Garden is in the city of'      → ' New'
'The Colosseum is in the city of'              → ' Rome'


Every invoke runs its own intervention code, but vLLM batches the underlying
forward passes for efficiency. You can pass **different sampling params per
invoke** too (e.g. `tracer.invoke(prompt, temperature=0.8)`).

### Collecting results across invokes

Variables defined at trace scope are shared across every invoke. That makes it
easy to build up a single data structure from many prompts:

In [6]:
prompts = [
    "The Eiffel Tower is in",
    "Madison Square Garden is in",
    "The Colosseum is in",
]

# Build the per-prompt containers up front, then .save() the list inside the trace.
all_tokens = [list() for _ in range(len(prompts))]

with model.trace(temperature=0.0, top_p=1, max_tokens=3) as tracer:
    # Shared, trace-scope state — .save() on the container itself.
    all_tokens = all_tokens.save()

    for i, prompt in enumerate(prompts):
        with tracer.invoke(prompt):
            # tracer.all() fires on every generation step.
            with tracer.all():
                all_tokens[i].append(model.samples.item())

for prompt, toks in zip(prompts, all_tokens):
    print(f"{prompt!r:<32} → {model.tokenizer.decode(toks)!r}")

Rendering prompts: 100%|█████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 2644.58it/s]


Processed prompts: 100%|████████████| 3/3 [00:00<00:00, 32.14it/s, est. speed input: 218.69 toks/s, output: 98.40 toks/s]

'The Eiffel Tower is in'         → ' Paris, France'
'Madison Square Garden is in'    → ' the heart of'
'The Colosseum is in'            → ' the center of'


Key points:

- **`.save()` the container, not the list contents.** The shared `all_tokens`
  list is saved once at trace scope; each invoke mutates it in place.
- **Per-invoke sampling params override the trace-level ones.** You can do
  `tracer.invoke(prompt, temperature=0.8)` inside a trace that was opened with
  `temperature=0.0`.

## Accessing logits and sampled tokens

`model.logits` and `model.samples` are synthesized envoys that sit at the end
of the pipeline:

- `model.logits` — the pre-sampling logits for the current generation
  step (shape `[vocab_size]` per invoke)
- `model.samples` — the scalar token id that was actually sampled

In [7]:
with model.trace(
    "Madison Square Garden is located in",
    temperature=0.8,
    top_p=0.95,
    max_tokens=3,
) as tracer:
    step_logits = list().save()
    step_samples = list().save()

    with tracer.all():
        step_logits.append(model.logits)
        step_samples.append(model.samples.item())

for i, (l, s) in enumerate(zip(step_logits, step_samples)):
    print(f"step {i}: top-1 via argmax={l.argmax().item():5d}  sampled={s:5d}"
          f"  ({model.tokenizer.decode(s)!r})")

Processed prompts:   0%|                       | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|█████████████| 1/1 [00:00<00:00,  8.40it/s, est. speed input: 60.29 toks/s, output: 25.83 toks/s]

step 0: top-1 via argmax=  279  sampled= 1532  (' New')
step 1: top-1 via argmax= 4261  sampled= 4261  (' York')
step 2: top-1 via argmax= 4311  sampled= 4311  (' City')


Because we're sampling with `temperature=0.8`, the sampled token often
doesn't match `argmax(logits)`. You can also **write** into `model.logits`
or `model.samples` to force specific sampling decisions.

## Multi-token generation with `.all()` and `.iter[...]`

`tracer.all()` applies its body to **every** generation step:

In [8]:
with model.trace("Hello world", max_tokens=5) as tracer:
    sampled = list().save()
    with tracer.all():
        sampled.append(model.samples.item())

print("tokens:", sampled)
print("decoded:", model.tokenizer.decode(sampled))

Rendering prompts: 100%|█████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1663.09it/s]


Processed prompts: 100%|█████████████| 1/1 [00:00<00:00,  9.57it/s, est. speed input: 19.72 toks/s, output: 49.30 toks/s]

tokens: [0, 358, 2776, 13295, 700]
decoded: ! I'm checking out


`tracer.iter[slice]` runs its body only on specific generation steps. This is
useful when you want to intervene on (or observe) a window of tokens:

In [9]:
prompt = "The Eiffel Tower is in the city of"
mlp = model.model.layers[6].mlp

with model.trace(prompt, max_tokens=6) as tracer:
    hidden_states = list().save()

    # Zero out the MLP output on steps 2-4 (inclusive-exclusive): clone, mutate
    # the clone, and assign it back through .output (not an in-place write).
    with tracer.iter[2:5]:
        masked = mlp.output.clone()
        masked[-1] = 0
        mlp.output = masked
        hidden_states.append(mlp.output)

print(f"captured {len(hidden_states)} steps")

Processed prompts:   0%|                       | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|█████████████| 1/1 [00:00<00:00,  8.20it/s, est. speed input: 84.12 toks/s, output: 50.46 toks/s]

captured 3 steps


Inside `tracer.all()` / `tracer.iter[...]` you can both **read** (append to a
saved list) and **write** (mutate `.output`). Writes apply only on the steps
the iterator selects.

## Efficient activation extraction with `tracer.cache()`

The patterns above all involve explicitly appending values to a saved list.
For dense per-layer activation capture — e.g. "give me the residual stream at
layer 6 for every token of every request" — NNsight provides
`tracer.cache()`, which registers a persistent hook on each target module and
collects activations into a `CacheDict` keyed by module path.

In [10]:
target_layer = model.model.layers[6]

with model.trace(
    "The Eiffel Tower is in the city of",
    temperature=0.0,
    max_tokens=8,
) as tracer:
    cache = tracer.cache(modules=[target_layer]).save()

import torch

print("cached module paths:", list(cache.keys()))

# The CacheDict is keyed by Envoy path. For Qwen under vLLM that's
# "model.model.layers.6" (the leading "model" is vLLM's wrapper namespace).
# When a module fires multiple times (prefill + each decode step) the entry
# is a list of Entry objects — concatenate their hidden states to get every
# captured token in one tensor.
entry = cache[next(iter(cache.keys()))]
entries = entry if isinstance(entry, list) else [entry]

def hidden_states(e):
    out = e.output
    # vLLM decoder layers return (mlp_output, residual); the full hidden state is their sum.
    return out[0] + out[1] if isinstance(out, tuple) else out

all_hs = torch.cat([hidden_states(e) for e in entries], dim=0)
print(f"captured {len(entries)} forward passes")
print(f"total hidden-state shape: {tuple(all_hs.shape)}  "
      f"(prefill tokens + one row per decode step)")

Processed prompts:   0%|                       | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|█████████████| 1/1 [00:00<00:00,  7.16it/s, est. speed input: 73.15 toks/s, output: 58.51 toks/s]

cached module paths: ['model.model.layers.6']
captured 8 forward passes
total hidden-state shape: (17, 896)  (prefill tokens + one row per decode step)


A few things to know about `tracer.cache()`:

- **Multiple modules.** Pass a list (`modules=[layer_a, layer_b]`) or leave it
  out to cache *every* module in the model.
- **Inputs too, if you want them.** `tracer.cache(include_inputs=True)`
  captures each module's `(args, kwargs)` alongside its output.
- **Device / dtype controls.** Pass `device=torch.device("cpu")` or
  `dtype=torch.float32` to move/cast the captured tensors as they're
  collected. CPU is the default.
- **Per-request transport.** Inside the async backend, the cache is zstd-
  compressed and pickled per request instead of in one giant blob, which
  matters a lot at high concurrency.

`tracer.cache()` is the API the
[`nnsight-vllm-lens-comparison`](https://github.com/JadenFiotto-Kaufman/nnsight-vllm-lens-comparison)
benchmark uses to extract residual streams at scale.

## Async mode (`mode="async"`)

Pass `mode="async"` to `VLLM(...)` to load the model against vLLM's
`AsyncLLM` engine instead of the sync `LLM`. The trace-writing API is the
same, but the **result** is an async generator that streams `RequestOutput`
objects as tokens land.

In [11]:
import nnsight as nns
from nnsight.modeling.vllm import VLLM

async_vllm = VLLM(
    "Qwen/Qwen2.5-0.5B",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.2,
    dispatch=True,
    mode="async",
)

async def run_one(prompt: str):
    with async_vllm.trace(prompt, temperature=0.0, max_tokens=5) as tracer:
        cache = tracer.cache(modules=[async_vllm.model.layers[6]])
        nns.save(cache)

    # tracer.backend is an AsyncVLLMBackend instance you iterate as an
    # async generator. Each yielded output is a vLLM RequestOutput with
    # a .saves dict attached once the request finishes.
    final = None
    async for output in tracer.backend:
        if output.finished:
            final = output

    return final

# Jupyter already runs an asyncio loop — just `await` directly. In a plain
# Python script you'd wrap this in `asyncio.run(run_one(...))`.
final = await run_one("The Eiffel Tower is in")
print("finished:", final.finished)
print("decoded:", final.outputs[0].text)
print("saves:", list(final.saves.keys()))

The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1

finished: True
decoded:  Paris, France. It
saves: ['cache']


The async path is the right choice when you're firing many requests
concurrently — e.g. using `asyncio.gather` to submit a whole dataset at once.
vLLM's scheduler forms multi-request batches, and each request's saves are
transported back independently (zstd-compressed) as it finishes, rather than
waiting on one giant pickle at the end.

Sync vs. async at a glance:

|                          | Sync (`mode="sync"`, default) | Async (`mode="async"`) |
|--------------------------|------------------------------|------------------------|
| Engine class             | `vllm.LLM`                   | `vllm.v1.engine.async_llm.AsyncLLM` |
| Best for                 | Notebook experiments, small batches | High-throughput concurrent workloads |
| Trace API                | `with model.trace(...) as t:` | same — but iterate `t.backend` after |
| Saves transport          | One pickle per finished request | zstd-compressed per-request saves on every streamed output |
| Multi-token intervention | `tracer.iter[...]`, `tracer.all()` | same |

## Tensor parallelism and multi-GPU

`VLLM(..., tensor_parallel_size=N)` shards the model across `N` GPUs on a
single node. NNsight handles the sharded-tensor semantics automatically:
intervention code always sees the **full** gathered tensor (not a shard),
and writes are re-sharded before being passed back to vLLM. From your
perspective as a user, `model.trace(...)` looks identical to the `tp=1`
case — you just see the full hidden-state tensor.

The code below isn't executed inline because this notebook runs on a single
GPU, but the pattern is drop-in: if you have two GPUs visible via
`CUDA_VISIBLE_DEVICES`, just set `tensor_parallel_size=2`.

<pre><code>from nnsight.modeling.vllm import VLLM

# 2-way tensor parallelism on a single machine
vllm_tp2 = VLLM(
    "facebook/opt-6.7b",
    tensor_parallel_size=2,
    gpu_memory_utilization=0.85,
    dispatch=True,
)

# Use it exactly like a tp=1 model — NNsight gathers sharded tensors
# before your intervention code runs and re-shards after.
with vllm_tp2.trace("Hello world", max_tokens=5) as tracer:
    mid = vllm_tp2.model.decoder.layers[16].output.save()
</code></pre>

Under the hood, NNsight's `VLLMBatcher` registers pre/post hooks on every
parallel linear layer (`ColumnParallelLinear`, `RowParallelLinear`). On entry
to a sharded layer it gathers the sharded input/output with
`tensor_model_parallel_all_gather`, runs the intervention code against the
full tensor on every rank, and on exit re-splits the tensor so vLLM's
forward pass sees the same shape it would have without the hooks. Every
GPU runs the same intervention on the same complete tensor.

### Multi-node with Ray

For tensor parallelism across multiple machines, switch the executor to Ray:

<pre><code>vllm_multinode = VLLM(
    "meta-llama/Llama-3.1-70B",
    tensor_parallel_size=8,
    distributed_executor_backend="ray",
    dispatch=True,
)
</code></pre>

Before instantiation, point `RAY_ADDRESS` at an existing Ray cluster's GCS
address (e.g. `RAY_ADDRESS=head-node:6379`, **not** `ray://host:port`).
NNsight's `NNsightRayExecutor` joins the cluster as a driver-only node and
places workers on the available machines. You don't need to change any of
your tracing code — interventions run identically against a multi-node Ray
executor.

A full multi-node Docker example, including the Dockerfile and cluster
test harness, lives at
[`nnsight/src/nnsight/modeling/vllm/examples/multi_node_with_ray/`](https://github.com/ndif-team/nnsight/tree/main/src/nnsight/modeling/vllm/examples/multi_node_with_ray).

## Limitations and gotchas

A short list of things that surprise people:

- **No gradients.** vLLM's paged-attention kernels don't retain a computation
  graph, so `.backward()`, `.grad`, and any gradient-based operations aren't
  supported on `VLLM` models. Use `LanguageModel` for those.
- **vLLM ≠ transformers numerically.** Fused kernels, different attention
  implementations, quantization defaults, and temperature-0 argmax being
  deterministic *within an engine* (but not *across* engines at different
  batch sizes) all mean you shouldn't expect exact-match outputs between
  `VLLM` and `LanguageModel` for the same input. The interventions are
  correct in both — the baselines just aren't identical.
- **Module access differs from HuggingFace.** vLLM's fused kernels and
  tensor-parallel layers change what a module returns, and reading the wrong
  thing usually *runs without error but is silently wrong*:
    - A decoder layer returns `(sub_layer_output, residual)`, not a combined
      hidden state — the full residual stream is
      `layer.output[0] + layer.output[1]` (this is why the `tracer.cache()`
      example above sums the two).
    - `layer.input` is the int64 **position IDs**; the hidden states are
      `layer.inputs[0][1]`.
    - The fused RMSNorm returns `(normalized, residual)` (use `norm.output[0]`),
      and `RowParallelLinear` (`o_proj`, `down_proj`) returns `(output, bias)`
      (use `.output[0]`).
    - QKV and gate/up are merged into `qkv_proj` / `gate_up_proj` — split the
      output with `.split([q_size, kv_size, kv_size], dim=-1)` /
      `.chunk(2, dim=-1)`; the separate projections don't exist.

  The full inventory with the fix for each lives in the repo's
  [`intervention-gaps/REPORT.md`](https://github.com/ndif-team/nnsight/blob/main/src/nnsight/modeling/vllm/intervention-gaps/REPORT.md).
- **Logits only on the last token.** vLLM runs `lm_head` only on each request's
  final token (all it needs to sample the next one), so `model.logits` covers
  just that position. To read logits at an *earlier* position (a logit lens),
  apply the final norm and unembedding to a saved hidden state **inside the
  trace** — the recipe is in
  [`intervention-gaps/VLLM_GUIDE.md`](https://github.com/ndif-team/nnsight/blob/main/src/nnsight/modeling/vllm/intervention-gaps/VLLM_GUIDE.md).
- **One prompt per invoke.** As noted above, `tracer.invoke(prompt)` takes
  exactly one string / token-id list. To batch, loop invokes inside a single
  trace; vLLM handles the batching.
- **`tracer.cache()` under preempt+recompute.** When 1000s of concurrent
  requests exceed the KV-cache budget, vLLM evicts some in-flight requests
  and **re-runs their full prefill** on resume. The cache hook fires for both
  the original run and the recompute, so those requests end up with
  ~2× their expected token rows in the `CacheDict`. If you're running at a
  scale where preemption can happen (check for the engine's `Preempted` log
  lines), provision enough `gpu_memory_utilization` / `max_num_seqs` to keep
  everything resident.
- **Async engine requires `asyncio`.** `mode="async"` traces can only be
  iterated inside a coroutine. In a notebook, use
  `await` directly (Jupyter runs an event loop for you) or `asyncio.run(...)`
  from a plain Python script.

## See also

**Other feature guides:**

- [Tracer fundamentals](1_getting.ipynb) — how traces, invokes, and `.save()`
  work in general
- [Multi-token generation](4_multiple_token.ipynb) — deeper coverage of
  `tracer.all()` / `tracer.iter[...]`

**End-to-end examples:**

- [`ndif-team/nnsight-vllm-demos`](https://github.com/ndif-team/nnsight-vllm-demos) —
  runnable demos of the vLLM integration, including a streaming chat UI and
  a Llama-Scope-SAE steering example that modifies activations mid-generation
  behind a real chat server
- [`JadenFiotto-Kaufman/nnsight-vllm-lens-comparison`](https://github.com/JadenFiotto-Kaufman/nnsight-vllm-lens-comparison) —
  reproducible benchmark pitting `tracer.cache()` against the vllm-lens
  plugin on the same OPT workload (OPT-30B, 1000 Alpaca prompts, 1024 max
  tokens, TP=2); includes a `validate.py` that bitwise-compares the
  extracted activations

**Contributor-facing internals:**

- [`ndif-team/nnsight`](https://github.com/ndif-team/nnsight) — source for
  the full vLLM integration
- [`src/nnsight/modeling/vllm/README.md`](https://github.com/ndif-team/nnsight/blob/main/src/nnsight/modeling/vllm/README.md) —
  architecture document: mediator transport via `extra_args`, batch-group
  management (flat-token vs. prompt-level), the three interleaving phases
  (forward / logits / sampling), tensor-parallel gather/reshard, the Ray
  distributed executor, and the async engine. Read this if you're modifying
  the integration or trying to understand why a specific intervention
  behaves the way it does inside vLLM.
- [`src/nnsight/modeling/vllm/examples/multi_node_with_ray/`](https://github.com/ndif-team/nnsight/tree/main/src/nnsight/modeling/vllm/examples/multi_node_with_ray) —
  Docker-based multi-node Ray example with a test harness for verifying TP
  across machines